In [1]:
import os
import json
import bisect
import math
from datetime import datetime
import yfinance as yf
import pandas as pd

In [ ]:
# Add directories
parent_root = 'yahoofinance-raw-data'
if not os.path.exists(parent_root):
        os.makedirs(parent_root)

In [2]:
def pull_mcap_to_annualized_metric_multiple(start_date, end_date, ticker, metric):
    # -----------------------------------
    # 1) Fetch the quarterly financials
    # -----------------------------------
    tkr = yf.Ticker(ticker)
    df  = tkr.quarterly_financials

    # debug: see what row labels actually exist
    print(f"[{ticker}] available quarterly_financials metrics:")
    print(df.index.tolist())

    # pick only the ones that exist
    desired = ["Total Revenue", "Operating Revenue", "EBITDA"]
    available = df.index.intersection(desired)
    if available.empty:
        raise KeyError(f"None of {desired} found in quarterly_financials for {ticker}")
    
    # pivot so dates → rows
    subset = df.loc[available].T

    # 2) Normalize index, fill NaNs
    subset.index = subset.index.strftime("%Y-%m-%d")
    subset      = subset.where(pd.notnull(subset), None)
    quarterly_metrics = subset.to_dict(orient="index")

    # save quarterly metrics JSON
    os.makedirs("yahoofinance-raw-data", exist_ok=True)
    qm_path = f"yahoofinance-raw-data/{ticker}_quarterly_metrics.json"
    with open(qm_path, "w") as f:
        json.dump(quarterly_metrics, f, indent=2)
    print(f"Wrote {qm_path}")

    # -----------------------------------
    # 3) Pull sharesOutstanding & history → market caps
    # -----------------------------------
    shares_out = tkr.info.get("sharesOutstanding")
    if shares_out is None:
        raise ValueError(f"Could not fetch sharesOutstanding for {ticker}")

    hist = tkr.history(start=start_date, end=end_date)
    hist["MarketCap"] = hist["Close"] * shares_out

    mc = {
        dt.strftime("%Y-%m-%d"): cap
        for dt, cap in zip(hist.index, hist["MarketCap"])
    }
    mc_path = f"yahoofinance-raw-data/{ticker}_market_cap.json"
    with open(mc_path, "w") as f:
        json.dump(mc, f, indent=2)
    print(f"Wrote {mc_path}")

    # -----------------------------------
    # 4) Build daily mcap/metric multiple
    # -----------------------------------
    quarter_strs  = sorted(quarterly_metrics.keys())
    quarter_dates = [datetime.strptime(q, "%Y-%m-%d") for q in quarter_strs]

    # re-download full history for daily multiples
    hist = tkr.history(start=quarter_strs[0],
                       end=datetime.today().strftime("%Y-%m-%d"))
    if hasattr(hist.index, "tz"):
        hist.index = hist.index.tz_localize(None)
    hist["MarketCap"] = hist["Close"] * shares_out

    daily_data = {}
    for dt, row in hist.iterrows():
        date_str = dt.strftime("%Y-%m-%d")
        idx      = bisect.bisect_right(quarter_dates, dt) - 1
        if idx < 0:
            continue
        q        = quarter_strs[idx]
        total_metric = quarterly_metrics[q].get(f"{metric}")  # use whichever metric you prefer
        if not total_metric:
            continue

        mcap = row["MarketCap"]
        ann_metric = total_metric * 4
        multiple = mcap / ann_metric
        daily_data[date_str] = {
            "market_cap": mcap,
            f"{metric}": total_metric,
            f"annualized {metric}": ann_metric,
            f"mcap_to_annualized_{metric}": multiple
        }

    # 5) 7‑day trailing average
    dates = sorted(daily_data)
    vals  = [daily_data[d][f"mcap_to_annualized_{metric}"] for d in dates]
    trailing = {}
    for i, d in enumerate(dates):
        if i < 6:
            trailing[d] = None
        else:
            trailing[d] = sum(vals[i-6:i+1]) / 7

    # 6) merge & sanitize
    results = {}
    for d in dates:
        results[d] = {
            f"{ticker}_market_cap": daily_data[d]["market_cap"],
            f"{ticker}_{metric}": daily_data[d][f"{metric}"],
            f"{ticker}_annualized_{metric}": daily_data[d][f"annualized {metric}"],
            f"{ticker}_mcap_to_annualized_{metric}": daily_data[d][f"mcap_to_annualized_{metric}"],
            f"{ticker}_7d_avg": trailing[d]
        }

    for metrics in results.values():
        for k, v in metrics.items():
            if isinstance(v, float) and math.isnan(v):
                metrics[k] = None

    # 7) save final JSON
    out_path = f"yahoofinance-raw-data/{ticker}_mcap_to_annualized_{metric}.json"
    with open(out_path, "w") as f:
        json.dump(results, f, indent=2)
    print(f"Wrote multiples to {out_path}")

In [3]:
# call the function
tickers   = ['COIN', 'CRCL', 'HOOD']
start_date = "2020-01-01"
end_date   = datetime.today().strftime("%Y-%m-%d")
metrics = ['Total Revenue', 'EBITDA']

for metric in metrics:
    print(f'{metric}')
    for ticker in tickers:
        print(f'{ticker}')
        pull_mcap_to_annualized_metric_multiple(start_date, end_date, ticker, metric)
        print('---------------------------')
    print()
print('Done')

Total Revenue
COIN
[COIN] available quarterly_financials metrics:
['Tax Effect Of Unusual Items', 'Tax Rate For Calcs', 'Normalized EBITDA', 'Total Unusual Items', 'Total Unusual Items Excluding Goodwill', 'Net Income From Continuing Operation Net Minority Interest', 'Reconciled Depreciation', 'Reconciled Cost Of Revenue', 'EBITDA', 'EBIT', 'Net Interest Income', 'Interest Expense', 'Normalized Income', 'Net Income From Continuing And Discontinued Operation', 'Total Expenses', 'Total Operating Income As Reported', 'Diluted Average Shares', 'Basic Average Shares', 'Diluted EPS', 'Basic EPS', 'Diluted NI Availto Com Stockholders', 'Average Dilution Earnings', 'Net Income Common Stockholders', 'Otherunder Preferred Stock Dividend', 'Net Income', 'Net Income Including Noncontrolling Interests', 'Net Income Continuous Operations', 'Tax Provision', 'Pretax Income', 'Other Income Expense', 'Other Non Operating Income Expenses', 'Special Income Charges', 'Other Special Charges', 'Write Off', '